# Ground-truth pace — interactive tuner

Reconstruct a clean per-second pace from Garmin running dynamics and **tune the estimator live** with sliders, using the `pace_predict` library.

The reconstruction segments the run by **cadence phase** (accelerating / steady / decelerating), holds each steady segment near-constant, ramps the transitions, interpolates short GNSS dropouts, and calibrates to the ≥1 km laps. The sliders expose every threshold (`PhaseParams` + `SegmentParams`) so you can fit it per runner.

> **Privacy.** This reads FIT files from `data/` (gitignored) and shows home GPS-derived data, so **run it locally**. Outputs are stripped before commit (nbstripout) — no GPS/health data is ever committed. CI verifies committed notebooks carry no outputs.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

from pace_predict.groundtruth import (
    ACCEL, DECEL, PhaseParams, SegmentParams,
    ground_truth_speed, integrate_speed, median_dt_s, phase_labels, phase_segments,
)
from pace_predict.inventory import build_inventory
from pace_predict.io import load_activity
from pace_predict.io import schema as S


def _repo_root() -> Path:
    """Repo root (dir with pyproject.toml), so the notebook works from any kernel cwd."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    return Path.cwd()


DATA = _repo_root() / 'data' / 'fit'


def to_pace(speed_mps):
    """m/s -> min/km, blanking near-stops so pauses don't dominate the axis."""
    v = np.asarray(speed_mps, float)
    v = np.where(v > 0.6, v, np.nan)
    return 1000.0 / v / 60.0

## Dataset inventory
One row per activity. `track` flags runs on a 400 m oval (GNSS overshoots — held out of calibration); `dyn_pace` marks the embedded `dynamicsPace` field, the baseline to beat.

In [ ]:
inventory = build_inventory(sorted(DATA.glob('*.fit')))
if inventory.empty:
    raise RuntimeError(f'No activities under {DATA} — add Garmin .fit files (see data/README.md).')
inventory[['file', 'sport', 'dur_min', 'dist_km', 'laps', 'track', 'cadence_spm', 'gct_ms', 'dyn_pace']]

## Interactive tuner

Pick a run and drag the sliders; the ground-truth line (blue) and its accel/decel markers update live. Grey relief along the bottom is the elevation profile; faint vertical lines are lap boundaries.

**Phase classification** — `thr_spm_s` (cadence-slope threshold), `smooth_s` (cadence smoothing), `min_phase_s` (shortest transition kept), `min_delta_spm` (smallest plateau-level change kept). **Reconstruction** — `max_drift_mps` (how much a steady segment may slope for a hill), `prefilter_s` (spike-reject median). **Dropouts** — `dropout_gap_s` (longest GNSS dropout interpolated).

What to look for: steady segments should sit flat at the right level, transitions should be crisp at each rep, and the calibration error should stay small (< ~1%).

In [ ]:
def compute(run, params):
    act = load_activity(DATA / run)
    r = act.records
    dt = median_dt_s(r)
    minutes = r[S.T_S].to_numpy() / 60.0
    gt = ground_truth_speed(act, params=params)
    labels = phase_labels(r[S.CADENCE_SPM].to_numpy(), dt, params=params.phase)
    alt = r[S.ALTITUDE_M].to_numpy()
    if np.isfinite(alt).any():
        lo, hi = np.nanmin(alt), np.nanmax(alt)
        alt_norm = (alt - lo) / (hi - lo) if hi > lo else np.zeros_like(alt)
    else:
        alt_norm = np.zeros_like(alt)
    start0 = r[S.TIMESTAMP].min()
    lap_min = [(e - start0).total_seconds() / 60.0 for e in act.laps[S.LAP_END_TIME].dropna()]

    dist_gt = float(integrate_speed(gt, dt)[-1])
    dist_rec = float(np.nanmax(r[S.DISTANCE_M].to_numpy()))
    has_dist = bool(np.isfinite(dist_rec) and dist_rec > 0)
    cal_pct = 100.0 * (dist_gt - dist_rec) / dist_rec if has_dist else float('nan')
    seg = phase_segments(labels)
    counts = {p: sum(1 for s in seg if s.phase == p) for p in (ACCEL, 0, DECEL)}
    steady_pct = 100.0 * float(np.mean(labels == 0))
    return {
        'minutes': minutes, 'raw_pace': to_pace(r[S.SPEED_MPS].to_numpy()),
        'gt_pace': to_pace(gt), 'labels': labels, 'alt_norm': alt_norm, 'lap_min': lap_min,
        'cal_pct': cal_pct, 'counts': counts, 'steady_pct': steady_pct,
    }


def _params_from(sliders):
    return SegmentParams(
        phase=PhaseParams(
            thr_spm_s=sliders['thr'].value, smooth_s=sliders['smooth'].value,
            min_phase_s=sliders['min_phase'].value, min_delta_spm=sliders['min_delta'].value,
        ),
        prefilter_s=sliders['prefilter'].value, max_drift_mps=sliders['max_drift'].value,
        dropout_gap_s=sliders['dropout'].value,
    )


def _lap_shapes(lap_min):
    return [dict(type='line', x0=x, x1=x, y0=0, y1=1, yref='paper',
                 line=dict(color='rgba(0,0,0,0.12)', width=1)) for x in lap_min]


# Widgets: run picker + one slider per tunable parameter.
runs = sorted(p.name for p in DATA.glob('*.fit'))
default_run = inventory.sort_values('laps', ascending=False)['file'].iloc[0]
run_dd = widgets.Dropdown(options=runs, value=default_run, description='run')
S_ = dict(style={'description_width': '110px'}, layout=widgets.Layout(width='320px'),
          continuous_update=False)
sliders = {
    'thr': widgets.FloatSlider(0.45, min=0.15, max=1.0, step=0.05, description='thr_spm_s', **S_),
    'smooth': widgets.FloatSlider(9, min=3, max=21, step=2, description='smooth_s', **S_),
    'min_phase': widgets.FloatSlider(3, min=1, max=10, step=1, description='min_phase_s', **S_),
    'min_delta': widgets.FloatSlider(2, min=0.5, max=6, step=0.5, description='min_delta_spm', **S_),
    'max_drift': widgets.FloatSlider(0.25, min=0.0, max=1.0, step=0.05, description='max_drift_mps', **S_),
    'prefilter': widgets.FloatSlider(7, min=1, max=21, step=2, description='prefilter_s', **S_),
    'dropout': widgets.FloatSlider(8, min=0, max=20, step=1, description='dropout_gap_s', **S_),
}
metrics = widgets.HTML()

fig = go.FigureWidget()
fig.add_trace(go.Scatter(yaxis='y2', fill='tozeroy', mode='none',
                         fillcolor='rgba(150,150,150,0.22)', hoverinfo='skip', showlegend=False))
fig.add_trace(go.Scatter(name='raw enhanced_speed', line=dict(color='lightgray', width=1)))
fig.add_trace(go.Scatter(name='ground truth', line=dict(color='royalblue', width=2.4)))
fig.add_trace(go.Scatter(name='accel', mode='markers', marker=dict(color='crimson', size=5)))
fig.add_trace(go.Scatter(name='decel', mode='markers', marker=dict(color='seagreen', size=5)))
fig.update_layout(height=520, margin=dict(t=40, b=40), xaxis=dict(title='time (min)'),
                  yaxis=dict(title='pace (min/km)', autorange='reversed'),
                  yaxis2=dict(overlaying='y', side='right', range=[0, 4], showgrid=False,
                              showticklabels=False, fixedrange=True),
                  legend=dict(orientation='h', y=1.08))


def update(_=None):
    d = compute(run_dd.value, _params_from(sliders))
    m, lab, gp = d['minutes'], d['labels'], d['gt_pace']
    with fig.batch_update():
        fig.data[0].x, fig.data[0].y = m, d['alt_norm']
        fig.data[1].x, fig.data[1].y = m, d['raw_pace']
        fig.data[2].x, fig.data[2].y = m, gp
        fig.data[3].x, fig.data[3].y = m[lab == ACCEL], gp[lab == ACCEL]
        fig.data[4].x, fig.data[4].y = m[lab == DECEL], gp[lab == DECEL]
        fig.layout.shapes = _lap_shapes(d['lap_min'])
        fig.layout.title.text = run_dd.value
        # Focus the pace axis on the running range (ignore the standing-start spike).
        finite = np.concatenate([d['raw_pace'], gp])
        finite = finite[np.isfinite(finite)]
        if finite.size:
            lo, hi = np.nanpercentile(finite, [2, 98])
            pad = 0.15 * (hi - lo) + 0.1
            fig.layout.yaxis.range = [hi + pad, lo - pad]  # reversed axis
    c = d['counts']
    metrics.value = (
        f"<b>calibration</b> {d['cal_pct']:+.2f}% vs recorded distance &nbsp;|&nbsp; "
        f"<b>steady</b> {d['steady_pct']:.0f}% of samples &nbsp;|&nbsp; "
        f"<b>segments</b> {c[0]} steady / {c[ACCEL]} accel / {c[DECEL]} decel"
    )


run_dd.observe(update, names='value')
for s in sliders.values():
    s.observe(update, names='value')
update()

controls = widgets.VBox([run_dd, *sliders.values()])
display(widgets.VBox([metrics, widgets.HBox([controls, fig])]))

## Per-lap validation
Over ≥1 km, GNSS pace is trustworthy, so the exact lap pace (distance/time) and `enhanced_speed` agree to a few s/km. Short reps diverge (GNSS error) — and cadence/GCT still separate work from recovery, which is what the model will exploit.

In [ ]:
def per_lap_table(act) -> pd.DataFrame:
    r = act.records
    t_s = r[S.T_S].to_numpy()
    start0 = r[S.TIMESTAMP].min()
    speed = r[S.SPEED_MPS].to_numpy()
    cadence = r[S.CADENCE_SPM].to_numpy()
    gct = r[S.GROUND_CONTACT_TIME_MS].to_numpy()

    def mean_int(values):  # None when a lap has no samples (e.g. GCT on a multisport bike leg)
        return None if not np.isfinite(values).any() else round(float(np.nanmean(values)))

    ends = [(e - start0).total_seconds() if not pd.isna(e) else float(t_s[-1])
            for e in act.laps[S.LAP_END_TIME]]
    starts = [0.0, *ends[:-1]]
    rows = []
    for i, (a, b) in enumerate(zip(starts, ends, strict=True)):
        mask = (t_s >= a) & (t_s < b)
        distance = act.laps[S.LAP_DISTANCE_M].iloc[i]
        duration = act.laps[S.LAP_ELAPSED_S].iloc[i]
        if mask.sum() == 0 or not (np.isfinite(distance) and distance > 0):
            continue
        if not (np.isfinite(duration) and duration > 0):
            continue
        rows.append({
            'lap': i, 'dist_m': round(distance),
            'exact_pace': round(1000.0 / (distance / duration) / 60.0, 2),
            'es_pace': round(float(np.nanmean(to_pace(speed[mask]))), 2),
            'cadence': mean_int(cadence[mask]),
            'gct_ms': mean_int(gct[mask]),
            'trusted_>=1km': distance >= 1000,
        })
    return pd.DataFrame(rows)


per_lap_table(load_activity(DATA / run_dd.value))